# BirdCLEF 2026 - Kaggle CPU优化版

## 竞赛要求适配
- ✅ 类别数：234物种（从sample_submission获取）
- ✅ 音频窗口：5秒
- ✅ 评估指标：Macro ROC-AUC
- ✅ 运行环境：CPU only（禁止GPU，90分钟限制）

## Kaggle路径配置
- 数据路径：`/kaggle/input/competitions/birdclef-2026/`
- 模型权重：`/kaggle/input/models/aiaiaiooo/tf-efficientnet-b0/pytorch/default/1/`

## 优化策略（适配90分钟限制）
1. 使用 EfficientNet-B0 轻量模型
2. 减少epoch数量（3轮，在90分钟内完成）
3. 批次大小16，加快训练
4. 预计算Mel频谱缓存，避免重复计算
5. 使用GroupKFold单折验证

## ⚠️ 重要提醒
- 使用CPU训练，单epoch约30-35分钟
- 3个epoch总耗时约90分钟，刚好符合限制
- 必须在90分钟内完成，否则超时

In [ ]:
import sys
import platform

print("="*60)
print("Python环境检测")
print("="*60)
print(f"Python版本: {sys.version}")
print(f"Python路径: {sys.executable}")
print(f"系统平台: {platform.platform()}")
print(f"处理器: {platform.processor()}")
print(f"架构: {platform.machine()}")
print("="*60)

# 检查已安装的包
try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
except:
    print("❌ PyTorch: 未安装")

try:
    import timm
    print(f"✅ timm: {timm.__version__}")
except:
    print("❌ timm: 未安装")

try:
    import tensorflow as tf
    print(f"✅ TensorFlow: {tf.__version__}")
except:
    print("❌ TensorFlow: 未安装")

try:
    import torchaudio
    print(f"✅ torchaudio: {torchaudio.__version__}")
except:
    print("❌ torchaudio: 未安装")

In [ ]:
# ========================================
# 1. 导入必要的库
# ========================================

import os  # 操作系统模块，用于文件路径操作
import sys  # 系统模块，用于刷新输出缓冲区
import random  # 随机模块，用于设置随机种子
import numpy as np  # NumPy库，用于数组运算
import pandas as pd  # Pandas库，用于读取CSV数据
import torch  # PyTorch深度学习框架
import torch.nn as nn  # 神经网络模块
import torchaudio  # 音频处理模块
from torch.utils.data import Dataset, DataLoader  # 数据集和数据加载器
from sklearn.model_selection import GroupKFold  # GroupKFold交叉验证
from sklearn.metrics import roc_auc_score  # AUC评估指标
import timm  # 预训练模型库
from tqdm import tqdm  # 进度条
import datetime  # 日期时间模块
import warnings  # 警告模块
warnings.filterwarnings('ignore')  # 忽略所有警告

print('✅ 所有库导入成功')  # 打印确认信息

In [ ]:
# ========================================
# 2. 检查设备
# ========================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 自动检测可用设备
print(f'🔧 使用设备: {device}')  # 打印当前设备

if torch.cuda.is_available():  # 如果有GPU
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')  # 打印GPU名称
else:  # 如果是CPU
    print('⚠️ 使用CPU训练，速度较慢，请耐心等待...')  # 打印警告

In [ ]:
# ========================================
# 3. 设置随机种子（保证可复现）
# ========================================

def set_seed(seed):  # 定义随机种子函数
    random.seed(seed)  # Python随机种子
    np.random.seed(seed)  # NumPy随机种子
    torch.manual_seed(seed)  # PyTorch随机种子
    if torch.cuda.is_available():  # 如果有GPU
        torch.cuda.manual_seed_all(seed)  # 所有GPU随机种子

set_seed(42)  # 设置随机种子为42

In [ ]:
import torch
import torch.nn as nn
import torchaudio

class Config:
    # ==================== Kaggle路径配置 ====================
    DATA_DIR = '/kaggle/input/competitions/birdclef-2026/'
    AUDIO_DIR = '/kaggle/input/competitions/birdclef-2026/train_audio/'
    MODEL_DIR = '/kaggle/input/models/aiaiaiooo/tf-efficientnet-b0/pytorch/default/1/'
    OUTPUT_DIR = '/kaggle/working/'

    # ==================== 随机种子 ====================
    seed = 42

    # ==================== 训练参数（适配90分钟限制） ====================
    epochs = 3
    batch_size = 16
    num_workers = 0
    lr = 1e-3
    weight_decay = 1e-4
    patience = 3

    # ==================== 模型配置 ====================
    model_name = 'tf_efficientnet_b0'
    num_classes = 234
    in_channels = 1

    # ==================== 音频处理参数 ====================
    duration = 5
    sr = 32000
    n_fft = 2048
    n_mels = 128
    f_min = 50
    f_max = 15000

    # ==================== 数据增强参数 ====================
    mix_alpha = 0.4
    cutmix_alpha = 0.4
    gradient_accumulation_steps = 2

    # ==================== SpecAugment参数 ====================
    freq_mask_param = 15
    time_mask_param = 15
    n_freq_masks = 2
    n_time_masks = 2

config = Config()

print('=' * 60)
print('Kaggle环境配置')
print('=' * 60)
print(f'   数据目录: {config.DATA_DIR}')
print(f'   模型目录: {config.MODEL_DIR}')
print(f'   输出目录: {config.OUTPUT_DIR}')
print(f'   模型: {config.model_name}')
print(f'   批次大小: {config.batch_size}')
print(f'   学习率: {config.lr}')
print(f'   训练轮数: {config.epochs}')
print(f'   类别数: {config.num_classes}')
print('=' * 60)

---

## 📊 数据处理

---


In [ ]:
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import soundfile as sf
import ast
import pickle

class BirdDataset(Dataset):
    def __init__(self, df, labels, train=True, config=None, cache_dir=None):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.config = config or Config()
        self.labels = labels
        self.num_classes = len(labels)
        self.label_to_idx = {l: i for i, l in enumerate(labels)}

        self.duration = self.config.duration
        self.sr = self.config.sr
        self.target_samples = int(self.duration * self.sr)

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=self.config.n_fft,
            f_min=self.config.f_min,
            f_max=self.config.f_max,
            n_mels=self.config.n_mels,
            hop_length=512
        )

        self.freq_mask = torchaudio.transforms.FrequencyMasking(
            freq_mask_param=self.config.freq_mask_param
        )
        self.time_mask = torchaudio.transforms.TimeMasking(
            time_mask_param=self.config.time_mask_param
        )

        # 尝试从缓存加载
        if cache_dir and os.path.exists(cache_dir):
            print(f'从缓存加载: {cache_dir}')
            with open(cache_dir, 'rb') as f:
                cache_data = pickle.load(f)
                self.cache = cache_data['features']
            print(f'已加载 {len(self.cache)} 个缓存样本')
        else:
            # 预计算所有Mel频谱
            self.cache = []
            print(f'预计算Mel频谱中，共 {len(self.df)} 个样本...')
            for idx in tqdm(range(len(self.df)), desc='预计算'):
                mel, label = self._load_and_compute(idx)
                self.cache.append((mel, label))
            print('✅ 预计算完成!')

            # 保存到缓存
            if cache_dir:
                os.makedirs(os.path.dirname(cache_dir), exist_ok=True)
                with open(cache_dir, 'wb') as f:
                    pickle.dump({'features': self.cache}, f)
                print(f'💾 缓存已保存: {cache_dir}')

    def _load_and_compute(self, idx):
        row = self.df.iloc[idx]
        filename = row.filename
        primary_label = row.primary_label

        # 尝试多种可能的路径
        possible_paths = [
            self.config.AUDIO_DIR + filename,
            self.config.AUDIO_DIR + primary_label + '/' + filename,
            self.config.AUDIO_DIR + filename.replace('.ogg', '.wav'),
            self.config.AUDIO_DIR + primary_label + '/' + filename.replace('.ogg', '.wav'),
        ]

        file_path = None
        for path in possible_paths:
            if os.path.exists(path):
                file_path = path
                break

        # 如果文件不存在，返回零向量
        if file_path is None:
            time_steps = int(self.config.duration * self.config.sr / 512)
            mel = torch.zeros(1, self.config.n_mels, time_steps)
            label = np.zeros(self.num_classes)
            return mel, label

        # 加载音频
        audio, sr = sf.read(file_path)
        audio = torch.from_numpy(audio).float()

        # 重采样
        if sr != self.sr:
            audio = torchaudio.functional.resample(audio, sr, self.sr)

        # 转单声道
        if len(audio.shape) > 1 and audio.shape[0] > 1:
            audio = audio.mean(axis=0)

        # 随机裁剪或填充
        if len(audio) > self.target_samples:
            if self.train:
                start = torch.randint(0, len(audio) - self.target_samples, (1,)).item()
            else:
                start = 0
            audio = audio[start:start + self.target_samples]
        elif len(audio) < self.target_samples:
            padding = self.target_samples - len(audio)
            audio = torch.nn.functional.pad(audio, (0, padding))

        # 转换为Mel频谱
        x = self.mel_transform(audio)
        x = x.clamp(min=1e-9).log()

        if len(x.shape) == 2:
            x = x.unsqueeze(0)

        # 标准化
        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True)
        x = (x - mean) / (std + 1e-6)

        # 创建标签向量
        label = np.zeros(self.num_classes)
        label[self.label_to_idx[primary_label]] = 1

        # 处理secondary_labels
        if 'secondary_labels' in row.index and pd.notna(row.get('secondary_labels')):
            try:
                secondary_str = str(row.get('secondary_labels', ''))
                if secondary_str and secondary_str not in ['', '[]', 'nan']:
                    secondary = ast.literal_eval(secondary_str)
                    for sec_label in secondary:
                        if sec_label in self.label_to_idx:
                            label[self.label_to_idx[sec_label]] = 1
            except (ValueError, SyntaxError, TypeError):
                pass

        return x, label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mel, label = self.cache[idx]
        mel = mel.clone()

        if self.train:
            for _ in range(self.config.n_freq_masks):
                mel = self.freq_mask(mel)
            for _ in range(self.config.n_time_masks):
                mel = self.time_mask(mel)

        return mel, torch.from_numpy(label).float()

---

## 🤖 模型定义

---


In [ ]:
class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or Config()

        print(f'创建模型: {self.config.model_name}')

        # 使用timm创建模型
        self.backbone = timm.create_model(
            self.config.model_name,
            pretrained=False,
            num_classes=0,
            in_chans=self.config.in_channels
        )

        # 从Kaggle数据集目录加载预训练权重
        model_dir = self.config.MODEL_DIR

        if os.path.exists(model_dir):
            files = os.listdir(model_dir)
            print(f'模型目录文件: {files}')

            # 查找权重文件
            weight_file = None
            for f in files:
                if any(ext in f.lower() for ext in ['.pth', '.pt', '.bin', '.safetensors']):
                    weight_file = os.path.join(model_dir, f)
                    break

            if weight_file:
                print(f'加载权重: {weight_file}')
                if weight_file.endswith('.safetensors'):
                    from safetensors.torch import load_file
                    state_dict = load_file(weight_file)
                else:
                    state_dict = torch.load(weight_file, map_location='cpu')

                # 过滤掉classifier相关的键
                state_dict = {k: v for k, v in state_dict.items() if 'classifier' not in k}

                # 如果权重是3通道，需要调整第一层卷积
                for key in list(state_dict.keys()):
                    if 'conv_stem.weight' in key and state_dict[key].shape[1] == 3:
                        state_dict[key] = state_dict[key][:, :1, :, :]
                        print('已将第一层卷积从3通道转换为1通道')

                missing, unexpected = self.backbone.load_state_dict(state_dict, strict=False)
                if missing:
                    print(f'缺失的键: {missing}')
                print(f'✅ 权重加载成功!')
            else:
                print(f'⚠️ 目录中没有找到权重文件，将使用随机初始化')
        else:
            print(f'⚠️ 模型目录不存在: {model_dir}')

        # 获取backbone输出维度
        with torch.no_grad():
            dummy = torch.randn(1, 1, 128, 313)
            feat = self.backbone(dummy)
            self.feat_dim = feat.shape[1]

        print(f'Backbone特征维度: {self.feat_dim}')

        # 添加自定义分类头
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.config.num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

In [ ]:
# ========================================
# 7. Mixup数据增强
# ========================================

class Mixup:  # Mixup数据增强类
    def __init__(self, alpha=0.4):  # 初始化函数
        self.alpha = alpha  # Mixup参数
        
    def __call__(self, x, y):  # 调用函数
        lam = np.random.beta(self.alpha, self.alpha)  # 从Beta分布采样混合比例
        batch_size = x.size(0)  # 获取批次大小
        index = torch.randperm(batch_size).to(x.device)  # 生成随机索引
        mixed_x = lam * x + (1 - lam) * x[index]  # 混合输入数据
        mixed_y = lam * y + (1 - lam) * y[index]  # 混合标签
        return mixed_x, mixed_y  # 返回混合后的数据和标签

# ========================================
# 7. Cutmix数据增强
# ========================================

class Cutmix:  # Cutmix数据增强类
    def __init__(self, alpha=0.4):  # 初始化函数
        self.alpha = alpha  # Cutmix参数
    
    def __call__(self, x, y):  # 调用函数
        lam = np.random.beta(self.alpha, self.alpha)  # 从Beta分布采样
        batch_size = x.size(0)  # 获取批次大小
        index = torch.randperm(batch_size).to(x.device)  # 生成随机索引
        
        # 随机裁剪区域
        W = x.size(3)  # 宽度
        H = x.size(2)  # 高度
        cut_rat = np.sqrt(1. - lam)  # 裁剪比例
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        
        # 随机裁剪位置
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        # 确保裁剪区域在边界内
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        
        # 执行Cutmix
        x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))  # 更新混合比例
        
        mixed_y = lam * y + (1 - lam) * y[index]  # 混合标签
        return x, mixed_y

---

## 🏋️ 训练器

---


In [ ]:
# ========================================
# 8. Trainer训练器类
# ========================================

class Trainer:  # 训练器类
    def __init__(self, train_loader, val_loader, config):  # 初始化函数
        self.train_loader = train_loader  # 训练数据加载器
        self.val_loader = val_loader  # 验证数据加载器
        self.config = config  # 配置参数
        self.device = device  # 计算设备
        
        # 创建模型
        self.model = BirdModel(config).to(self.device)  # 创建模型并移动到设备
        total_params = sum(p.numel() for p in self.model.parameters())  # 计算总参数量
        print(f'模型参数量: {total_params:,}')  # 打印参数量
        
        # 创建优化器
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),  # 模型参数
            lr=config.lr,  # 学习率
            weight_decay=config.weight_decay  # 权重衰减
        )
        
        # 学习率调度器（余弦退火+预热）
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer,  # 优化器
            T_0=5,  # 重启周期
            T_mult=2,  # 周期倍增
            eta_min=1e-6  # 最小学习率
        )
        
        # 损失函数
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=torch.ones(config.num_classes) * 2.0)  # BCEWithLogitsLoss损失
        
        # Mixup和Cutmix增强
        self.mixup = Mixup(alpha=config.mix_alpha)  # 创建Mixup实例
        self.cutmix = Cutmix(alpha=config.cutmix_alpha)  # 创建Cutmix实例
        
        # 最佳指标
        self.best_auc = 0  # 最佳AUC初始化为0
        self.best_epoch = 0  # 最佳epoch初始化为0
        
        # 早停计数器
        self.early_stop_count = 0
        
        # 实验目录
        self.exp_dir = None  # 实验目录初始化

    def create_experiment_dir(self):  # 创建实验目录
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')  # 获取时间戳
        self.exp_dir = f'{self.config.OUTPUT_DIR}{timestamp}_CPU优化版/'  # 实验目录路径
        os.makedirs(self.exp_dir, exist_ok=True)  # 创建目录
        print(f'实验目录: {self.exp_dir}')  # 打印目录路径

    def train_one_epoch(self, epoch):  # 训练一个epoch
        self.model.train()  # 设置为训练模式
        total_loss = 0  # 总损失初始化为0
        self.optimizer.zero_grad()  # 清空梯度

        pbar = tqdm(enumerate(self.train_loader), total=len(self.train_loader), desc=f'Epoch {epoch+1}')  # 创建进度条
        
        for batch_idx, (x, y) in pbar:  # 遍历每个批次
            x, y = x.to(self.device), y.to(self.device)  # 数据移到设备
            
            # 暂时关闭Mixup/Cutmix数据增强
            
            logits = self.model(x)  # 前向传播
            loss = self.criterion(logits, y)  # 计算损失
            
            # 梯度累积
            loss = loss / self.config.gradient_accumulation_steps  # 损失除以累积步数
            loss.backward()  # 反向传播
            
            if (batch_idx + 1) % self.config.gradient_accumulation_steps == 0:  # 达到累积步数
                self.optimizer.step()  # 更新参数
                self.optimizer.zero_grad()  # 清空梯度
            
            total_loss += loss.item() * self.config.gradient_accumulation_steps  # 累加损失
            pbar.set_postfix({'loss': f'{loss.item() * self.config.gradient_accumulation_steps:.4f}'})  # 显示损失
        
        return total_loss / len(self.train_loader)  # 返回平均损失

    def validate(self):  # 验证函数
        self.model.eval()  # 设置为评估模式
        total_loss = 0  # 总损失初始化为0
        preds = []  # 预测列表
        targets = []  # 标签列表

        with torch.no_grad():  # 禁用梯度计算
            for x, y in tqdm(self.val_loader, desc='Validation'):  # 遍历验证集
                x, y = x.to(self.device), y.to(self.device)  # 数据移到设备
                logits = self.model(x)  # 前向传播
                loss = self.criterion(logits, y)  # 计算损失
                total_loss += loss.item()  # 累加损失
                preds.append(torch.sigmoid(logits).cpu().numpy())  # 保存预测概率
                targets.append(y.cpu().numpy())  # 保存标签
        
        preds = np.concatenate(preds)  # 合并预测
        targets = np.concatenate(targets)  # 合并标签
        
        # 计算AUC
        aucs = []  # AUC列表
        for i in range(targets.shape[1]):  # 遍历每个类别
            if targets[:, i].sum() > 0:  # 如果有正样本
                try:  # 尝试计算AUC
                    auc = roc_auc_score(targets[:, i], preds[:, i])  # 计算AUC
                    aucs.append(auc)  # 保存AUC
                except:  # 如果计算失败
                    pass  # 跳过
        
        avg_auc = np.mean(aucs) if aucs else 0  # 计算平均AUC
        avg_loss = total_loss / len(self.val_loader)  # 计算平均损失
        
        return avg_loss, avg_auc  # 返回损失和AUC

    def train(self):  # 主训练函数
        self.create_experiment_dir()  # 创建实验目录
        
        print('\n' + '='*60)  # 打印分隔线
        print('开始训练 (CPU优化版)')  # 打印标题
        print('='*60)  # 打印分隔线
        
        # 存储历史记录用于对比
        history = []
        
        for epoch in range(self.config.epochs):  # 遍历每个epoch
            epoch_start = datetime.datetime.now()  # 记录epoch开始时间
            print(f'\nEpoch {epoch+1}/{self.config.epochs}')  # 打印epoch信息
            
            current_lr = self.optimizer.param_groups[0]['lr']  # 获取当前学习率
            print(f'   学习率: {current_lr:.6f}')  # 打印学习率
            
            train_loss = self.train_one_epoch(epoch)  # 训练一个epoch
            val_loss, val_auc = self.validate()  # 验证模型
            
            epoch_end = datetime.datetime.now()  # 记录epoch结束时间
            epoch_duration = epoch_end - epoch_start  # 计算epoch耗时
            
            # 计算预计剩余时间
            remaining_epochs = self.config.epochs - epoch - 1
            estimated_remaining = epoch_duration * remaining_epochs
            
            # 保存历史记录
            history.append({
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_auc': val_auc,
                'duration': epoch_duration
            })
            
            # 与上一轮对比
            if epoch > 0:
                train_loss_diff = train_loss - history[epoch-1]['train_loss']
                val_loss_diff = val_loss - history[epoch-1]['val_loss']
                val_auc_diff = val_auc - history[epoch-1]['val_auc']
                
                train_loss_trend = '↓' if train_loss_diff < 0 else '↑' if train_loss_diff > 0 else '→'
                val_loss_trend = '↓' if val_loss_diff < 0 else '↑' if val_loss_diff > 0 else '→'
                val_auc_trend = '↑' if val_auc_diff > 0 else '↓' if val_auc_diff < 0 else '→'
                
                train_loss_diff_str = f' ({train_loss_trend} {abs(train_loss_diff):.4f})'
                val_loss_diff_str = f' ({val_loss_trend} {abs(val_loss_diff):.4f})'
                val_auc_diff_str = f' ({val_auc_trend} {abs(val_auc_diff):.4f})'
            else:
                train_loss_diff_str = ''
                val_loss_diff_str = ''
                val_auc_diff_str = ''
            
            # 打印结果
            print(f'   训练损失: {train_loss:.4f}{train_loss_diff_str}')  # 打印训练损失
            print(f'   验证损失: {val_loss:.4f}{val_loss_diff_str}')  # 打印验证损失
            print(f'   验证AUC:  {val_auc:.4f}{val_auc_diff_str}')  # 打印验证AUC
            print(f'   耗时:     {str(epoch_duration)[:-7]} (预计剩余: {str(estimated_remaining)[:-7]})')
            
            if val_auc > self.best_auc:  # 如果当前AUC是最佳
                self.best_auc = val_auc  # 更新最佳AUC
                self.best_epoch = epoch + 1  # 更新最佳epoch
                torch.save(self.model.state_dict(), self.exp_dir + 'best_model.pth')  # 保存最佳模型
                print(f'   保存最佳模型! AUC: {self.best_auc:.4f}')  # 打印信息
                self.early_stop_count = 0  # 重置早停计数器
            else:
                self.early_stop_count += 1  # 增加早停计数器
            
            # 早停检查
            if self.early_stop_count >= self.config.patience:
                print(f'   早停触发! 连续{self.config.patience}轮无改进')
                break
            
            # 更新学习率调度器
            self.scheduler.step()
        
        print('\n' + '='*60)  # 打印分隔线
        print('训练完成!')  # 打印完成信息
        print(f'最佳验证AUC: {self.best_auc:.4f} (Epoch {self.best_epoch})')  # 打印最佳AUC
        print('='*60)  # 打印分隔线
        
        return self.model  # 返回训练好的模型

---

## 📥 加载数据

---


In [ ]:
# ========================================
# 9. 加载训练数据
# ========================================

print('📂 加载训练数据...')

# 读取训练数据CSV
train_df = pd.read_csv(config.DATA_DIR + 'train.csv')
print(f'✅ 训练样本总数: {len(train_df)}')

# 读取sample_submission获取完整的物种列表（234个物种）
sample_sub = pd.read_csv(config.DATA_DIR + 'sample_submission.csv')
submission_cols = [c for c in sample_sub.columns if c != 'row_id']
labels = sorted(submission_cols)
print(f'🦜 类别数量: {len(labels)} (从sample_submission获取)')

# 训练数据中有但提交列表中没有的物种
train_species = set(train_df['primary_label'].unique())
missing_species = [s for s in train_species if s not in labels]
if missing_species:
    print(f'⚠️ 训练数据中有 {len(missing_species)} 个物种不在提交列表中')

# 创建GroupKFold划分
if 'folded' not in train_df.columns:
    gkf = GroupKFold(n_splits=5)
    train_df['folded'] = -1
    for fold_idx, (_, val_idx) in enumerate(gkf.split(train_df, groups=train_df['primary_label'])):
        train_df.loc[val_idx, 'folded'] = fold_idx

# 使用第0折作为验证集
fold = 0
train_fold_df = train_df[train_df.folded != fold].reset_index(drop=True)
val_fold_df = train_df[train_df.folded == fold].reset_index(drop=True)

print(f'📊 训练集: {len(train_fold_df)} 样本')
print(f'📊 验证集: {len(val_fold_df)} 样本')

In [ ]:
# ========================================
# 10. 创建数据集和数据加载器
# ========================================

print('\n🔧 创建训练数据集...')
train_cache = config.OUTPUT_DIR + 'train_features.pkl'
train_dataset = BirdDataset(train_fold_df, labels, train=True, config=config, cache_dir=train_cache)

print('\n🔧 创建验证数据集...')
val_cache = config.OUTPUT_DIR + 'val_features.pkl'
val_dataset = BirdDataset(val_fold_df, labels, train=False, config=config, cache_dir=val_cache)

# 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=False
)

print(f'\n📦 训练批次数: {len(train_loader)}')
print(f'📦 验证批次数: {len(val_loader)}')

---

## 🎯 开始训练

---


In [ ]:
# ========================================
# 11. 创建训练器并开始训练
# ========================================

trainer = Trainer(train_loader, val_loader, config)  # 创建训练器实例
model = trainer.train()  # 开始训练

print(f'\n📁 模型保存位置: {trainer.exp_dir}best_model.pth')  # 打印模型保存位置

---

## 📊 训练结果分析

---


In [ ]:
# ========================================
# 12. 训练结果分析 & 测试集推理
# ========================================

print('=' * 50)
print('📊 训练结果分析')
print('=' * 50)

print(f'\n🏆 最佳验证AUC: {trainer.best_auc:.4f}')
print(f'📅 最佳Epoch: {trainer.best_epoch}')

print('\n' + '=' * 50)
print('🔮 开始测试集推理')
print('=' * 50)

# 加载sample_submission获取测试集信息
sample_sub = pd.read_csv(config.DATA_DIR + 'sample_submission.csv')
submission_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'📋 提交列数: {len(submission_cols)}')

# 查找测试音频目录
test_paths = [
    '/kaggle/input/competitions/birdclef-2026/test_soundscapes/',
    '/kaggle/input/competitions/birdclef-2026/test_audio/',
    '/kaggle/input/competitions/birdclef-2026/test/',
]

test_audio_dir = None
for path in test_paths:
    if os.path.exists(path):
        test_audio_dir = path
        break

if test_audio_dir:
    test_files = os.listdir(test_audio_dir)
    print(f'🎵 测试音频目录: {test_audio_dir}')
    print(f'🎵 测试音频文件数: {len(test_files)}')
    test_audio_ids = [f.replace('.wav', '').replace('.ogg', '') for f in test_files]
else:
    print('⚠️ 测试音频目录不存在，使用sample_submission创建测试数据')
    test_audio_ids = sample_sub['row_id'].tolist()

# 创建测试数据集
class TestDataset(Dataset):
    def __init__(self, audio_ids, config=None, audio_dir=None):
        self.audio_ids = audio_ids
        self.config = config or Config()
        self.audio_dir = audio_dir or config.DATA_DIR

        self.duration = self.config.duration
        self.sr = self.config.sr
        self.target_samples = int(self.duration * self.sr)

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=self.config.n_fft,
            f_min=self.config.f_min,
            f_max=self.config.f_max,
            n_mels=self.config.n_mels,
            hop_length=512
        )

    def __len__(self):
        return len(self.audio_ids)

    def __getitem__(self, idx):
        audio_id = self.audio_ids[idx]

        # 尝试加载音频
        possible_paths = []
        for ext in ['.wav', '.ogg']:
            possible_paths.append(self.audio_dir + audio_id + ext)

        audio = None
        for path in possible_paths:
            if os.path.exists(path):
                try:
                    audio, sr = sf.read(path)
                    break
                except:
                    pass

        if audio is None:
            time_steps = int(self.config.duration * self.config.sr / 512)
            mel = torch.zeros(1, self.config.n_mels, time_steps)
            return audio_id, mel

        audio = torch.from_numpy(audio).float()

        if len(audio.shape) > 1 and audio.shape[0] > 1:
            audio = audio.mean(axis=0)

        if len(audio) > self.target_samples:
            audio = audio[:self.target_samples]
        elif len(audio) < self.target_samples:
            padding = self.target_samples - len(audio)
            audio = torch.nn.functional.pad(audio, (0, padding))

        x = self.mel_transform(audio)
        x = x.clamp(min=1e-9).log()

        if len(x.shape) == 2:
            x = x.unsqueeze(0)

        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True)
        x = (x - mean) / (std + 1e-6)

        return audio_id, x

# 创建测试数据集
print('\n🔧 创建测试数据集...')
test_dataset = TestDataset(test_audio_ids, config=config, audio_dir=test_audio_dir)
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)
print(f'📦 测试批次数: {len(test_loader)}')

# 推理
print('\n🚀 运行推理...')
trainer.model.eval()
results = {}

with torch.no_grad():
    for audio_ids, x in tqdm(test_loader, desc='推理中'):
        x = x.to(device)
        logits = trainer.model(x)
        probs = torch.sigmoid(logits).cpu().numpy()

        for i, audio_id in enumerate(audio_ids):
            results[audio_id] = probs[i]

print(f'✅ 推理完成! 共处理 {len(results)} 个样本')

# 生成提交文件
print('\n📝 生成提交文件...')

submission_data = []

for row_id in tqdm(sample_sub['row_id'].tolist(), desc='构建提交'):
    row = {'row_id': row_id}

    if row_id in results:
        pred = results[row_id]
        for i, col in enumerate(submission_cols):
            if i < len(pred):
                row[col] = pred[i]
            else:
                row[col] = 0.0
    else:
        for col in submission_cols:
            row[col] = 0.0

    submission_data.append(row)

submission_df = pd.DataFrame(submission_data)
final_cols = ['row_id'] + submission_cols
submission_df = submission_df[final_cols]

for col in submission_cols:
    if col not in submission_df.columns:
        submission_df[col] = 0.0
    else:
        submission_df[col] = submission_df[col].fillna(0.0)

submission_df.to_csv(config.OUTPUT_DIR + 'submission.csv', index=False)

print('\n' + '=' * 60)
print('✅ 提交文件已生成!')
print('=' * 60)
print(f'📁 路径: {config.OUTPUT_DIR}submission.csv')
print(f'📊 形状: {submission_df.shape}')
print('=' * 60)